# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide to loading and exploring the FAIR² dataset using the `mlcroissant` library. The analysis focuses on mass spectrometry data from human mast cell extracellular vesicles, provided in the Croissant open dataset format via SenScience.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display high-level metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities (record sets, fields, columns, etc.) are referenced by their `@id` as per the Croissant schema.

In [ ]:
# List all available record sets and their fields using their @id

def describe_record_sets(ds):
    print("Record Sets available in the dataset (by @id):\n")
    for record_set in ds.record_sets:
        print(f"- Record Set @id: {record_set['@id']}")
        fields = record_set.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields/Columns (@id):")
        for field in fields:
            if isinstance(field, str):
                print(f"    - {field}")
            elif '@id' in field:
                print(f"    - {field['@id']}")
        print("")

describe_record_sets(dataset)


Below we print several records from the main record set for a quick overview.

*Replace `main_record_set_id` with the `@id` of the desired record set from the list above.*

In [ ]:
# Example - Print sample records from a record set

# Set this variable to the @id of the principal data record set (inspect previous output)
main_record_set_id = 'cr:recordSet/0'

try:
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        print(rec)
        if i >= 2:
            break
except Exception as e:
    print(f"Could not iterate the records for {main_record_set_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List of relevant record set @id's (as found in section 2)
# Replace with actual @ids from "describe_record_sets" output if needed

record_sets = ['cr:recordSet/0']  # update this list if more record sets are relevant
dataframes = {}

for rs_id in record_sets:
    print(f"\nLoading data for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Fields (@id) in DataFrame: {df.columns.tolist()}")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

**Note:** Update the `numeric_field_id` and `group_field_id` by referencing the field @ids from the DataFrame columns above.

In [ ]:
# Choose a numeric field (e.g., peptide counts, coverage, MW, etc.) by its @id.
numeric_field_id = 'cr:field/peptide_count'  # Replace with actual @id for a numeric field
group_field_id = 'cr:field/modification'     # Replace with a meaningful @id to group by, if available

rs_df = dataframes['cr:recordSet/0']

# Show summary statistics for numeric fields
if numeric_field_id in rs_df.columns:
    print(f"Quick stats for {numeric_field_id}:")
    print(rs_df[numeric_field_id].describe())
    # Filter out records with numeric_field_id > threshold
    threshold = rs_df[numeric_field_id].mean()  # or set an absolute value like 10
    filtered_df = rs_df[rs_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally group by another field if present
    if group_field_id in rs_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    else:
        print(f"Column {group_field_id} not found.\nAvailable columns: {rs_df.columns.tolist()}")
else:
    print(f"Column {numeric_field_id} not found.\nAvailable columns: {rs_df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Visualize distribution of a numeric field
if numeric_field_id in rs_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(rs_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

# Example: Boxplot by a group field
if (numeric_field_id in rs_df.columns) and (group_field_id in rs_df.columns):
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=rs_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
In this notebook, we have demonstrated how to load and explore the FAIR² mass spectrometry dataset using `mlcroissant`. We loaded the metadata, inspected available record sets and their fields (by `@id`), extracted the main data table, performed basic EDA, and visualized numeric fields and relationships between attributes.

**Key findings:**
- The data allows analysis of protein abundance and modification across extracellular vesicles isolated from human mast cells.
- Filtering and normalization steps are straightforward, facilitating downstream proteomics or biomarker discovery analysis.

**Next steps:**
- Apply domain-specific statistical tests or ML models.
- Explore additional record sets, if available, or map fields to controlled vocabularies for integration with other datasets.
- Reference all entities by `@id` (as shown) to ensure reproducibility across FAIR and Croissant-compliant workflows.